# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the [FAIR² Mass Spectrometry dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa) using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure the mlcroissant library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset with mlcroissant
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, their fields, and the corresponding `@id`s (Croissant IDs).

Let's enumerate all record sets defined in the schema, along with their fields and columns.

In [ ]:
# Retrieve and display information about each record set and its fields/columns
record_sets = list(dataset.record_sets)
print(f"Record Sets found: {len(record_sets)}\n")
for rs in record_sets:
    print(f"RecordSet @id: {rs.id}")
    print(f"  Name: {rs.name}")
    print(f"  Description: {getattr(rs, 'description', '')}")
    print(f"  Fields:")
    for field in rs.fields:
        print(f"    - @id: {field.id}  (name: {field.name})")
    print(f"  Columns:")
    for col in rs.columns:
        print(f"    - @id: {col.id}  (name: {col.name})")
    print("-")

## 3. Data Extraction
Load data from one or more record sets into pandas DataFrames for analysis. Use the record set `@id`s from the overview above below.

> **Note:** ALL access to record sets, fields, and columns should be done using their `@id`.

In [ ]:
# Identify the record set IDs for extraction
record_set_ids = [rs.id for rs in dataset.record_sets]
print("Record set @ids found:", record_set_ids)

# Load each record set as a pandas DataFrame
dataframes = {}
for record_set_id in record_set_ids:
    # The record_set_id is the '@id' within the dataset
    records = list(dataset.records(record_set=record_set_id))
    if len(records) > 0:
        dataframes[record_set_id] = pd.DataFrame(records)

if dataframes:
    # Choose the first record set for demonstration
    example_record_set_id = list(dataframes.keys())[0]
    print(f"Example DataFrame columns for record set '{example_record_set_id}':")
    print(dataframes[example_record_set_id].columns.tolist())
    dataframes[example_record_set_id].head()
else:
    print("No DataFrames loaded. Please check dataset availability.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and grouping by key fields.

In this section, we will:
- Filter records with a high peptide count (or another numeric field)
- Normalize a numeric field
- Group by a category if available

**Replace field `@id` variables below as appropriate for your chosen record set.**

> See the printout above for field (column) names and their exact `@id` values.

In [ ]:
# Assuming we use the first available record set for demonstration
record_set_id = example_record_set_id
df = dataframes[record_set_id]
print(f"Working with record set: {record_set_id} (columns: {df.columns.tolist()})\n")

# Identify a numeric field for EDA, e.g. 'peptide_count' or similar (adjust as needed)
possible_numeric_fields = [col for col in df.columns if df[col].dtype.kind in 'fi' and not col.startswith('@')]
print("Numeric fields detected:", possible_numeric_fields)
if possible_numeric_fields:
    numeric_field_id = possible_numeric_fields[0]
else:
    # Fallback: Try to find any field with 'count', 'abundance', or 'MW' in the name
    match_cols = [col for col in df.columns if any(s in col.lower() for s in ['count', 'abundance', 'mw', 'mass'])]
    numeric_field_id = match_cols[0] if match_cols else df.columns[0]

# Filter (example: keep only rows with value > 10)
threshold = 10
if numeric_field_id in df.columns:
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())
    # Normalize
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
    # Group by a categorical field (use accession/description/sample if present)
    possible_group_fields = [col for col in df.columns if df[col].dtype == object and col != numeric_field_id]
    if possible_group_fields:
        group_field = possible_group_fields[0]
        grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
        print(f"Grouped data by {group_field} (showing first few groups):")
        print(grouped_df.head())
    else:
        print("No suitable categorical field found for grouping.")
else:
    print(f"Numeric field '{numeric_field_id}' not found in columns.")

## 5. Visualization
Visualize the data distribution or relationships between fields. We'll use pandas and matplotlib for simple plots.

Here, we plot the distribution of the chosen numeric field and, if available, its mean by group.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if possible_numeric_fields:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    # If grouping was performed
    if 'grouped_df' in locals():
        plt.figure(figsize=(10, 4))
        grouped_df[numeric_field_id].sort_values(ascending=False).head(20).plot(kind='bar')
        plt.title(f"Mean {numeric_field_id} by group ({group_field}) -- top 20")
        plt.ylabel(f"{numeric_field_id} (mean)")
        plt.xlabel(group_field)
        plt.show()

## 6. Conclusion
In this notebook, we've demonstrated how to use the `mlcroissant` library to load, explore, and analyze a Croissant-described dataset using only the dataset's schema URL and referencing every entity by its `@id`.

- **Metadata access:** We loaded and inspected the full dataset metadata.
- **Schema navigation:** We listed available record sets, fields, and columns, all by `@id`.
- **Data extraction:** We loaded records as DataFrames and explored the column structure.
- **EDA:** We filtered, normalized, grouped, and visualized key numeric fields.

You can adapt this notebook to work with other Croissant datasets by specifying a different schema URL and adjusting field/grouping variables as needed.